# NLP Lab 5: Text Representation using Statistical and Vector Space Models
CSET346 | Q1 to Q4

In [1]:
import re, math
import numpy as np
import pandas as pd
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
for r in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(r, quiet=True)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
STOP = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [t for t in word_tokenize(text) if t not in STOP and len(t) > 1]

## Q1. Term Frequency and TF-IDF from Scratch
**Formulas**
- TF(t, d) = count(t, d) / total terms in d
- IDF(t) = log10(N / df(t))
- TF-IDF(t, d) = TF(t, d) x IDF(t)

In [2]:
corpus1 = [
    "Machine learning is a branch of artificial intelligence. It lets computers learn from data. Machine learning models improve with more data.",
    "Deep learning uses neural networks with many layers. Neural networks learn complex patterns from large data. Deep models need powerful GPUs.",
    "Natural language processing helps computers understand human language. Text data is converted into numbers. Language models predict the next word.",
    "Cricket is a popular sport in India. The team scored many runs in the match. The bowler took five wickets and the team won.",
    "Football players train daily to improve fitness. The team won the league after a thrilling match. The striker scored many goals.",
    "Python is a popular programming language. Programmers write code to build software. Debugging code helps programmers fix errors.",
]
docs1 = ["Doc" + str(i + 1) for i in range(len(corpus1))]
proc1 = [preprocess(d) for d in corpus1]
pd.DataFrame({"Original": corpus1, "Processed": [" ".join(t) for t in proc1]}, index=docs1)

,Original,Processed
Doc1,Machine learning is a branch of artificial int...,machine learning branch artificial intelligenc...
Doc2,Deep learning uses neural networks with many l...,deep learning uses neural networks many layers...
Doc3,Natural language processing helps computers un...,natural language processing helps computers un...
Doc4,Cricket is a popular sport in India. The team ...,cricket popular sport india team scored many r...
Doc5,Football players train daily to improve fitnes...,football players train daily improve fitness t...
Doc6,Python is a popular programming language. Prog...,python popular programming language programmer...


In [3]:
for d, t in zip(docs1, proc1):
    print(d, t)

Doc1 ['machine', 'learning', 'branch', 'artificial', 'intelligence', 'lets', 'computers', 'learn', 'data', 'machine', 'learning', 'models', 'improve', 'data']
Doc2 ['deep', 'learning', 'uses', 'neural', 'networks', 'many', 'layers', 'neural', 'networks', 'learn', 'complex', 'patterns', 'large', 'data', 'deep', 'models', 'need', 'powerful', 'gpus']
Doc3 ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'text', 'data', 'converted', 'numbers', 'language', 'models', 'predict', 'next', 'word']
Doc4 ['cricket', 'popular', 'sport', 'india', 'team', 'scored', 'many', 'runs', 'match', 'bowler', 'took', 'five', 'wickets', 'team']
Doc5 ['football', 'players', 'train', 'daily', 'improve', 'fitness', 'team', 'league', 'thrilling', 'match', 'striker', 'scored', 'many', 'goals']
Doc6 ['python', 'popular', 'programming', 'language', 'programmers', 'write', 'code', 'build', 'software', 'debugging', 'code', 'helps', 'programmers', 'fix', 'errors']


In [4]:
vocab1 = sorted(set(w for t in proc1 for w in t))
print("Vocabulary size:", len(vocab1))
print(vocab1)

Vocabulary size: 66
['artificial', 'bowler', 'branch', 'build', 'code', 'complex', 'computers', 'converted', 'cricket', 'daily', 'data', 'debugging', 'deep', 'errors', 'fitness', 'five', 'fix', 'football', 'goals', 'gpus', 'helps', 'human', 'improve', 'india', 'intelligence', 'language', 'large', 'layers', 'league', 'learn', 'learning', 'lets', 'machine', 'many', 'match', 'models', 'natural', 'need', 'networks', 'neural', 'next', 'numbers', 'patterns', 'players', 'popular', 'powerful', 'predict', 'processing', 'programmers', 'programming', 'python', 'runs', 'scored', 'software', 'sport', 'striker', 'team', 'text', 'thrilling', 'took', 'train', 'understand', 'uses', 'wickets', 'word', 'write']


In [5]:
tf1 = pd.DataFrame(0.0, index=vocab1, columns=docs1)
for d, t in zip(docs1, proc1):
    c = Counter(t)
    for w in vocab1:
        tf1.loc[w, d] = c[w] / len(t)
print("TF matrix shape (vocab x docs):", tf1.shape)
tf1.round(4)

TF matrix shape (vocab x docs): (66, 6)


,Doc1,Doc2,Doc3,Doc4,Doc5,Doc6
artificial,0.0714,0.0000,0.0000,0.0000,0.0,0.0000
bowler,0.0000,0.0000,0.0000,0.0714,0.0,0.0000
branch,0.0714,0.0000,0.0000,0.0000,0.0,0.0000
build,0.0000,0.0000,0.0000,0.0000,0.0,0.0667
code,0.0000,0.0000,0.0000,0.0000,0.0,0.1333
...,...,...,...,...,...,...
understand,0.0000,0.0000,0.0588,0.0000,0.0,0.0000
uses,0.0000,0.0526,0.0000,0.0000,0.0,0.0000
wickets,0.0000,0.0000,0.0000,0.0714,0.0,0.0000
word,0.0000,0.0000,0.0588,0.0000,0.0,0.0000


In [6]:
N = len(proc1)
df1 = pd.Series({w: sum(w in t for t in proc1) for w in vocab1})
idf1 = np.log10(N / df1)
idf_table = pd.DataFrame({"df": df1, "IDF": idf1.round(4)})
print("IDF shape:", idf_table.shape)
idf_table

IDF shape: (66, 2)


,df,IDF
artificial,1,0.7782
bowler,1,0.7782
branch,1,0.7782
build,1,0.7782
code,1,0.7782
...,...,...
understand,1,0.7782
uses,1,0.7782
wickets,1,0.7782
word,1,0.7782


In [7]:
tfidf1 = tf1.mul(idf1, axis=0)
print("TF-IDF matrix shape (vocab x docs):", tfidf1.shape)
tfidf1.round(4)

TF-IDF matrix shape (vocab x docs): (66, 6)


,Doc1,Doc2,Doc3,Doc4,Doc5,Doc6
artificial,0.0556,0.000,0.0000,0.0000,0.0,0.0000
bowler,0.0000,0.000,0.0000,0.0556,0.0,0.0000
branch,0.0556,0.000,0.0000,0.0000,0.0,0.0000
build,0.0000,0.000,0.0000,0.0000,0.0,0.0519
code,0.0000,0.000,0.0000,0.0000,0.0,0.1038
...,...,...,...,...,...,...
understand,0.0000,0.000,0.0458,0.0000,0.0,0.0000
uses,0.0000,0.041,0.0000,0.0000,0.0,0.0000
wickets,0.0000,0.000,0.0000,0.0556,0.0,0.0000
word,0.0000,0.000,0.0458,0.0000,0.0,0.0000


In [8]:
for d in docs1:
    top = tfidf1[d].sort_values(ascending=False).head(5)
    print(d, "->", ", ".join(f"{w} ({v:.4f})" for w, v in top.items()))

Doc1 -> machine (0.1112), learning (0.0682), lets (0.0556), artificial (0.0556), branch (0.0556)
Doc2 -> neural (0.0819), deep (0.0819), networks (0.0819), powerful (0.0410), patterns (0.0410)
Doc3 -> language (0.0842), converted (0.0458), text (0.0458), understand (0.0458), human (0.0458)
Doc4 -> team (0.0682), bowler (0.0556), sport (0.0556), cricket (0.0556), india (0.0556)
Doc5 -> football (0.0556), daily (0.0556), striker (0.0556), thrilling (0.0556), fitness (0.0556)
Doc6 -> programmers (0.1038), code (0.1038), python (0.0519), write (0.0519), errors (0.0519)


## Q2. Vector Space Model and Document Similarity

In [9]:
corpus2 = [
    "The new smartphone features a faster processor and a brighter display.",
    "The new laptop features a faster processor and a longer battery life.",
    "Researchers built an artificial intelligence system that writes software code.",
    "The cricket team won the final match by five wickets.",
    "The striker scored twice as the football team won the league match.",
    "Tennis champion won the tournament after a long and tiring final.",
    "The student council organised a workshop on programming for university students.",
    "The university announced new courses on machine learning and artificial intelligence.",
    "Tech companies invest heavily in artificial intelligence and machine learning chips.",
    "The coach praised the players after the team won the championship match.",
]
docs2 = ["D" + str(i + 1) for i in range(len(corpus2))]
proc2 = [" ".join(preprocess(d)) for d in corpus2]
pd.DataFrame({"Original": corpus2, "Processed": proc2}, index=docs2)

,Original,Processed
D1,The new smartphone features a faster processor...,new smartphone features faster processor brigh...
D2,The new laptop features a faster processor and...,new laptop features faster processor longer ba...
D3,Researchers built an artificial intelligence s...,researchers built artificial intelligence syst...
D4,The cricket team won the final match by five w...,cricket team final match five wickets
D5,The striker scored twice as the football team ...,striker scored twice football team league match
D6,Tennis champion won the tournament after a lon...,tennis champion tournament long tiring final
D7,The student council organised a workshop on pr...,student council organised workshop programming...
D8,The university announced new courses on machin...,university announced new courses machine learn...
D9,Tech companies invest heavily in artificial in...,tech companies invest heavily artificial intel...
D10,The coach praised the players after the team w...,coach praised players team championship match


In [10]:
tv = TfidfVectorizer()
X2 = tv.fit_transform(proc2)
dtm = pd.DataFrame(X2.toarray(), index=docs2, columns=tv.get_feature_names_out())
print("Document-term matrix shape (docs x terms):", dtm.shape)
dtm.round(3)

Document-term matrix shape (docs x terms): (10, 55)


,announced,artificial,battery,brighter,built,champion,championship,chips,coach,code,companies,council,courses,cricket,display,faster,features,final,five,football,heavily,intelligence,invest,laptop,league,...,match,new,organised,players,praised,processor,programming,researchers,scored,smartphone,software,striker,student,students,system,team,tech,tennis,tiring,tournament,twice,university,wickets,workshop,writes
D1,0.000,0.000,0.000,0.418,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.418,0.355,0.355,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.311,0.000,0.000,0.000,0.355,0.000,0.000,0.000,0.418,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.000,0.000,0.386,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.328,0.328,0.000,0.000,0.000,0.000,0.000,0.000,0.386,0.000,...,0.000,0.287,0.000,0.000,0.000,0.328,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D3,0.000,0.279,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.279,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.375,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375
D4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.455,0.000,0.000,0.000,0.387,0.455,0.000,0.000,0.000,0.000,0.000,0.000,...,0.338,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.338,0.000,0.000,0.000,0.000,0.000,0.000,0.455,0.000,0.000
D5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.405,0.000,0.000,0.000,0.000,0.405,...,0.301,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.405,0.000,0.000,0.405,0.000,0.000,0.000,0.301,0.000,0.000,0.000,0.000,0.405,0.000,0.000,0.000,0.000
D6,0.000,0.000,0.000,0.000,0.000,0.418,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.355,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.418,0.418,0.418,0.000,0.000,0.000,0.000,0.000
D7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.386,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.386,0.000,0.000,0.000,0.386,0.000,0.000,0.000,0.000,0.000,0.386,0.386,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.328,0.000,0.386,0.000
D8,0.414,0.308,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.414,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.308,0.000,0.000,0.000,...,0.000,0.308,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.352,0.000,0.000,0.000
D9,0.000,0.271,0.000,0.000,0.000,0.000,0.000,0.364,0.000,0.000,0.364,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.364,0.271,0.364,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.364,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D10,0.000,0.000,0.000,0.000,0.000,0.000,0.443,0.000,0.443,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.329,0.000,0.000,0.443,0.443,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.329,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [11]:
for d in docs2[:3]:
    v = dtm.loc[d]
    print(d, "vector length", len(v), "| non-zero", int((v > 0).sum()))
    print(v[v > 0].round(3).to_dict(), "\n")

D1 vector length 55 | non-zero 7
{'brighter': 0.418, 'display': 0.418, 'faster': 0.355, 'features': 0.355, 'new': 0.311, 'processor': 0.355, 'smartphone': 0.418} 

D2 vector length 55 | non-zero 8
{'battery': 0.386, 'faster': 0.328, 'features': 0.328, 'laptop': 0.386, 'life': 0.386, 'longer': 0.386, 'new': 0.287, 'processor': 0.328} 

D3 vector length 55 | non-zero 8
{'artificial': 0.279, 'built': 0.375, 'code': 0.375, 'intelligence': 0.279, 'researchers': 0.375, 'software': 0.375, 'system': 0.375, 'writes': 0.375} 



In [12]:
S2 = cosine_similarity(X2)
sim2 = pd.DataFrame(S2, index=docs2, columns=docs2)
sim2.round(3)

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
D1,1.000,0.439,0.000,0.000,0.000,0.000,0.000,0.096,0.000,0.000
D2,0.439,1.000,0.000,0.000,0.000,0.000,0.000,0.088,0.000,0.000
D3,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.172,0.151,0.000
D4,0.000,0.000,0.000,1.000,0.204,0.137,0.000,0.000,0.000,0.223
D5,0.000,0.000,0.000,0.204,1.000,0.000,0.000,0.000,0.000,0.198
D6,0.000,0.000,0.000,0.137,0.000,1.000,0.000,0.000,0.000,0.000
D7,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.115,0.000,0.000
D8,0.096,0.088,0.172,0.000,0.000,0.000,0.115,1.000,0.385,0.000
D9,0.000,0.000,0.151,0.000,0.000,0.000,0.000,0.385,1.000,0.000
D10,0.000,0.000,0.000,0.223,0.198,0.000,0.000,0.000,0.000,1.000


In [13]:
pairs = [("D1", "D2"), ("D4", "D5"), ("D8", "D9"), ("D3", "D9"), ("D1", "D4")]
for a, b in pairs:
    print(f"cos({a}, {b}) = {sim2.loc[a, b]:.4f}")

cos(D1, D2) = 0.4388
cos(D4, D5) = 0.2037
cos(D8, D9) = 0.3846
cos(D3, D9) = 0.1510
cos(D1, D4) = 0.0000


In [14]:
upper = [(docs2[i], docs2[j], S2[i, j]) for i in range(len(docs2)) for j in range(i + 1, len(docs2))]
top_pairs = sorted(upper, key=lambda x: -x[2])[:5]
for a, b, s in top_pairs:
    print(f"{a} & {b}: {s:.4f}")
    print("  ", corpus2[int(a[1:]) - 1])
    print("  ", corpus2[int(b[1:]) - 1])

D1 & D2: 0.4388
   The new smartphone features a faster processor and a brighter display.
   The new laptop features a faster processor and a longer battery life.
D8 & D9: 0.3846
   The university announced new courses on machine learning and artificial intelligence.
   Tech companies invest heavily in artificial intelligence and machine learning chips.
D4 & D10: 0.2228
   The cricket team won the final match by five wickets.
   The coach praised the players after the team won the championship match.
D4 & D5: 0.2037
   The cricket team won the final match by five wickets.
   The striker scored twice as the football team won the league match.
D5 & D10: 0.1981
   The striker scored twice as the football team won the league match.
   The coach praised the players after the team won the championship match.


**How VSM converts text to vectors**

1. Each unique term in the corpus becomes one axis of a high-dimensional space.
2. Each document becomes a point/vector whose coordinate on an axis is the weight (count or TF-IDF) of that term in the document.
3. All documents share the same axes, so the corpus becomes a matrix of shape (documents x vocabulary).
4. Cosine similarity, cos(theta) = (A . B) / (|A| |B|), measures the angle between vectors, so length differences do not matter.

**Observation:** Pairs sharing content words (match/team/won for sports, artificial/intelligence/chips for tech) score high, while cross-topic pairs score near 0. Similarity is purely lexical: documents on the same topic with different wording score low.

## Q3. Distributional Representation and Word Vectors

In [15]:
sentences3 = [
    "the cat drinks milk", "the dog drinks water", "the cat chases the mouse", "the dog chases the cat",
    "the cat sleeps on the sofa", "the dog sleeps on the floor", "the mouse eats cheese", "the cat eats fish",
    "the dog eats meat", "the horse eats grass", "the cow drinks water", "the cow eats grass",
    "the horse runs in the field", "the dog runs in the park", "the cat runs in the garden",
    "the boy plays football in the park", "the girl plays cricket in the field", "the boy reads a book",
    "the girl reads a novel", "the teacher reads a book", "the student writes code", "the programmer writes code",
    "the student reads a book", "the programmer debugs code", "the teacher writes a book",
    "the boy drinks milk", "the girl drinks water", "the boy eats bread", "the girl eats cheese",
    "the programmer drinks coffee", "the teacher drinks coffee", "the student drinks water",
    "the boy plays cricket in the park", "the girl plays football in the garden",
    "the horse drinks water", "the cow drinks milk", "the student plays cricket", "the programmer plays football",
    "the teacher reads a novel", "the girl writes a novel",
]
print("Sentences:", len(sentences3))
tok3 = [word_tokenize(s.lower()) for s in sentences3]
print(tok3[:3])

Sentences: 40
[['the', 'cat', 'drinks', 'milk'], ['the', 'dog', 'drinks', 'water'], ['the', 'cat', 'chases', 'the', 'mouse']]


In [16]:
freq3 = Counter(w for s in tok3 for w in s)
vocab3 = sorted(freq3)
idx3 = {w: i for i, w in enumerate(vocab3)}
print("Vocabulary size:", len(vocab3))
print(freq3.most_common(15))

Vocabulary size: 41
[('the', 51), ('drinks', 10), ('eats', 7), ('in', 7), ('a', 7), ('cat', 6), ('plays', 6), ('girl', 6), ('dog', 5), ('water', 5), ('boy', 5), ('reads', 5), ('book', 4), ('teacher', 4), ('student', 4)]


In [17]:
WINDOW = 2
cooc = np.zeros((len(vocab3), len(vocab3)))
for s in tok3:
    for i, w in enumerate(s):
        for j in range(max(0, i - WINDOW), min(len(s), i + WINDOW + 1)):
            if i != j:
                cooc[idx3[w], idx3[s[j]]] += 1
cooc_df = pd.DataFrame(cooc, index=vocab3, columns=vocab3)
print("Co-occurrence matrix shape:", cooc_df.shape)
cooc_df.iloc[:12, :12].astype(int)

Co-occurrence matrix shape: (41, 41)


,a,book,boy,bread,cat,chases,cheese,code,coffee,cow,cricket,debugs
a,0,4,1,0,0,0,0,0,0,0,0,0
book,4,0,0,0,0,0,0,0,0,0,0,0
boy,1,0,0,1,0,0,0,0,0,0,1,0
bread,0,0,1,0,0,0,0,0,0,0,0,0
cat,0,0,0,0,0,2,0,0,0,0,0,0
chases,0,0,0,0,2,0,0,0,0,0,0,0
cheese,0,0,0,0,0,0,0,0,0,0,0,0
code,0,0,0,0,0,0,0,0,0,0,0,1
coffee,0,0,0,0,0,0,0,0,0,0,0,0
cow,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
selected = ["cat", "dog", "horse", "cow", "boy", "girl", "student", "programmer", "drinks", "eats", "reads", "writes", "book", "code"]
print("Selected words and frequency:", {w: freq3[w] for w in selected})
ctx_words = [w for w in vocab3 if w != "the"]
total = cooc.sum()
rs = cooc.sum(1, keepdims=True)
cs = cooc.sum(0, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    pmi = np.log((cooc * total) / (rs * cs))
ppmi = np.nan_to_num(np.maximum(pmi, 0), neginf=0, posinf=0)
wv = pd.DataFrame(ppmi, index=vocab3, columns=vocab3)
sel_vecs = wv.loc[selected]
print("Selected word vectors shape:", sel_vecs.shape)
sel_vecs.round(2).iloc[:, :12]

Selected words and frequency: {'cat': 6, 'dog': 5, 'horse': 3, 'cow': 3, 'boy': 5, 'girl': 6, 'student': 4, 'programmer': 4, 'drinks': 10, 'eats': 7, 'reads': 5, 'writes': 4, 'book': 4, 'code': 3}
Selected word vectors shape: (14, 41)


,a,book,boy,bread,cat,chases,cheese,code,coffee,cow,cricket,debugs
cat,0.00,0.00,0.00,0.00,0.00,2.04,0.00,0.00,0.00,0.00,0.00,0.00
dog,0.00,0.00,0.00,0.00,0.00,1.47,0.00,0.00,0.00,0.00,0.00,0.00
horse,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
cow,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
boy,0.51,0.00,0.00,2.86,0.00,0.00,0.00,0.00,0.00,0.00,1.25,0.00
girl,1.02,0.00,0.00,0.00,0.00,0.00,1.98,0.00,0.00,0.00,1.07,0.00
student,0.73,0.00,0.00,0.00,0.00,0.00,0.00,1.98,0.00,0.00,1.47,0.00
programmer,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.68,2.39,0.00,0.00,2.68
drinks,0.00,0.00,0.15,0.00,0.03,0.00,0.00,0.00,2.17,1.36,0.00,0.00
eats,0.00,0.00,0.51,2.52,0.38,0.00,2.52,0.00,0.00,1.02,0.00,0.00


In [19]:
ws = pd.DataFrame(cosine_similarity(sel_vecs.values), index=selected, columns=selected)
ws.round(2)

,cat,dog,horse,cow,boy,girl,student,programmer,drinks,eats,reads,writes,book,code
cat,1.00,0.46,0.13,0.23,0.19,0.03,0.02,0.01,0.21,0.27,0.01,0.01,0.00,0.00
dog,0.46,1.00,0.30,0.17,0.03,0.13,0.15,0.02,0.17,0.29,0.01,0.01,0.00,0.00
horse,0.13,0.30,1.00,0.75,0.05,0.19,0.23,0.02,0.25,0.29,0.00,0.01,0.00,0.00
cow,0.23,0.17,0.75,1.00,0.28,0.17,0.23,0.03,0.50,0.26,0.00,0.01,0.00,0.00
boy,0.19,0.03,0.05,0.28,1.00,0.37,0.27,0.14,0.24,0.30,0.06,0.04,0.17,0.00
girl,0.03,0.13,0.19,0.17,0.37,1.00,0.54,0.21,0.19,0.26,0.15,0.10,0.38,0.11
student,0.02,0.15,0.23,0.23,0.27,0.54,1.00,0.44,0.24,0.01,0.10,0.42,0.42,0.16
programmer,0.01,0.02,0.02,0.03,0.14,0.21,0.44,1.00,0.26,0.01,0.00,0.33,0.10,0.45
drinks,0.21,0.17,0.25,0.50,0.24,0.19,0.24,0.26,1.00,0.10,0.06,0.08,0.00,0.08
eats,0.27,0.29,0.29,0.26,0.30,0.26,0.01,0.01,0.10,1.00,0.02,0.01,0.00,0.00


In [20]:
for w in selected:
    s = ws[w].drop(w).sort_values(ascending=False).head(3)
    print(f"{w:11s} -> " + ", ".join(f"{k} ({v:.2f})" for k, v in s.items()))

cat         -> dog (0.46), eats (0.27), cow (0.23)


dog         -> cat (0.46), horse (0.30), eats (0.29)
horse       -> cow (0.75), dog (0.30), eats (0.29)
cow         -> horse (0.75), drinks (0.50), boy (0.28)
boy         -> girl (0.37), eats (0.30), cow (0.28)
girl        -> student (0.54), book (0.38), boy (0.37)
student     -> girl (0.54), programmer (0.44), book (0.42)
programmer  -> code (0.45), student (0.44), writes (0.33)
drinks      -> cow (0.50), programmer (0.26), horse (0.25)
eats        -> boy (0.30), horse (0.29), dog (0.29)
reads       -> writes (0.73), book (0.30), girl (0.15)
writes      -> reads (0.73), student (0.42), programmer (0.33)
book        -> student (0.42), girl (0.38), reads (0.30)
code        -> programmer (0.45), writes (0.23), book (0.19)


In [21]:
fn = {"the", "a", "in", "on"}
tok3_w2v = [[w for w in s if w not in fn] for s in tok3]
w2v = Word2Vec(tok3_w2v, vector_size=30, window=2, min_count=1, sg=1, negative=5, epochs=100, seed=42, workers=1)
for w in ["cat", "boy", "drinks", "book"]:
    print(f"Co-occurrence : {w:6s} ->", [k for k in ws[w].drop(w).sort_values(ascending=False).index[:3]] if w in ws else "")
    print(f"Word2Vec      : {w:6s} ->", [(k, round(v, 2)) for k, v in w2v.wv.most_similar(w, topn=3)])

Co-occurrence : cat    ->

 ['dog', 'eats', 'cow']
Word2Vec      : cat    -> [('dog', 0.46), ('drinks', 0.37), ('cow', 0.35)]
Co-occurrence : boy    -> ['girl', 'eats', 'cow']
Word2Vec      : boy    -> [('horse', 0.4), ('girl', 0.21), ('sleeps', 0.18)]
Co-occurrence : drinks -> ['cow', 'programmer', 'horse']
Word2Vec      : drinks -> [('sleeps', 0.37), ('cat', 0.37), ('milk', 0.31)]
Co-occurrence : book   -> ['student', 'girl', 'reads']
Word2Vec      : book   -> [('reads', 0.37), ('floor', 0.3), ('chases', 0.29)]


**Distributional hypothesis:** words that occur in similar contexts have similar meanings (Firth: "You shall know a word by the company it keeps").

Here `cat`, `dog`, `horse`, `cow` all appear after `the` and next to verbs like `eats`, `drinks`, `runs`, so their context vectors are close. `boy`, `girl`, `student`, `programmer` share `plays`, `reads`, `writes`. Each word is represented by its co-occurrence row (PPMI weighted), and cosine similarity of those rows reflects similarity of meaning. The optional extension uses Word2Vec trained on the same corpus (dense 30-d vectors instead of sparse count-based vectors); with a corpus this tiny, pre-trained GloVe or Word2Vec would give better quality.

## Q4. Sentence and Document-to-Vector Representation

In [22]:
sent4 = [
    "The football team won the match yesterday",
    "The cricket team won the final match",
    "Players train hard before the match",
    "The coach praised the team after the match",
    "Neural networks learn patterns from large data",
    "Machine learning models learn from training data",
    "Deep learning models need large data",
    "Programmers write code for learning models",
    "The chef cooked fresh pasta with tomato sauce",
    "The chef baked a sweet chocolate cake",
    "The restaurant serves fresh spicy food",
    "Fresh vegetables and fruits make healthy food",
]
topic4 = ["sports"] * 4 + ["ai_tech"] * 4 + ["food"] * 4
ids4 = ["S" + str(i + 1) for i in range(len(sent4))]
proc4 = [" ".join(preprocess(s)) for s in sent4]
pd.DataFrame({"Sentence": sent4, "Topic": topic4, "Processed": proc4}, index=ids4)

,Sentence,Topic,Processed
S1,The football team won the match yesterday,sports,football team match yesterday
S2,The cricket team won the final match,sports,cricket team final match
S3,Players train hard before the match,sports,players train hard match
S4,The coach praised the team after the match,sports,coach praised team match
S5,Neural networks learn patterns from large data,ai_tech,neural networks learn patterns large data
S6,Machine learning models learn from training data,ai_tech,machine learning models learn training data
S7,Deep learning models need large data,ai_tech,deep learning models need large data
S8,Programmers write code for learning models,ai_tech,programmers write code learning models
S9,The chef cooked fresh pasta with tomato sauce,food,chef cooked fresh pasta tomato sauce
S10,The chef baked a sweet chocolate cake,food,chef baked sweet chocolate cake


In [23]:
tv4 = TfidfVectorizer()
T4 = tv4.fit_transform(proc4)
tfidf_sent = pd.DataFrame(T4.toarray(), index=ids4, columns=tv4.get_feature_names_out())
print("TF-IDF sentence matrix shape:", tfidf_sent.shape)
tfidf_sent.iloc[:5, :12].round(3)

TF-IDF sentence matrix shape:

 (12, 44)


,baked,cake,chef,chocolate,coach,code,cooked,cricket,data,deep,final,food
S1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.0,0.000,0.0
S2,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.574,0.000,0.0,0.574,0.0
S3,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.0,0.000,0.0
S4,0.0,0.0,0.0,0.0,0.574,0.0,0.0,0.000,0.000,0.0,0.000,0.0
S5,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.338,0.0,0.000,0.0


In [24]:
tok4 = [p.split() for p in proc4]
try:
    import gensim.downloader as api
    wvec = api.load("glove-wiki-gigaword-50")
    emb_name = "pre-trained GloVe 50d"
except Exception:
    big = [preprocess(x) for x in corpus1 + corpus2 + sent4] + [[w for w in s if w not in {"the", "a", "in", "on"}] for s in tok3]
    wvec = Word2Vec(big * 20, vector_size=50, window=3, min_count=1, sg=1, epochs=30, seed=42, workers=1).wv
    emb_name = "Word2Vec trained on local corpora"
print("Embedding used:", emb_name)
def avg_vec(tokens):
    vs = [wvec[t] for t in tokens if t in wvec]
    return np.mean(vs, axis=0) if vs else np.zeros(wvec.vector_size)
A4 = np.vstack([avg_vec(t) for t in tok4])
print("Average word-vector sentence matrix shape:", A4.shape)
pd.DataFrame(A4[:3], index=ids4[:3]).round(3).iloc[:, :10]

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

...

[=================================================-] 100.0% 66.0/66.0MB downloaded

[=================================================-] 100.0% 66.0/66.0MB downloaded

[==================================================] 100.0% 66.0/66.0MB downloaded

Embedding used: pre-trained GloVe 50d
Average word-vector sentence matrix shape: (12, 50)


,0,1,2,3,4,5,6,7,8,9
S1,-0.829,0.020,-0.509,0.691,0.114,-0.666,-0.734,0.624,-0.529,-0.309
S2,-0.722,0.088,-0.882,0.871,0.001,-0.335,-0.342,0.929,0.048,-0.427
S3,-0.280,0.017,0.151,0.110,-0.172,-0.221,-0.371,0.721,-0.328,0.099


In [25]:
S_tfidf = cosine_similarity(T4)
S_avg = cosine_similarity(A4)
pairs4 = [(0, 1), (0, 3), (4, 5), (4, 6), (8, 9), (8, 10), (0, 4), (7, 11)]
rows = [(ids4[a], ids4[b], round(S_tfidf[a, b], 3), round(S_avg[a, b], 3)) for a, b in pairs4]
pd.DataFrame(rows, columns=["Sent A", "Sent B", "Cos (TF-IDF)", "Cos (Avg W2V)"])

,Sent A,Sent B,Cos (TF-IDF),Cos (Avg W2V)
0,S1,S2,0.342,0.947
1,S1,S4,0.342,0.937
2,S5,S6,0.277,0.854
3,S5,S7,0.277,0.909
4,S9,S10,0.147,0.884
5,S9,S11,0.120,0.833
6,S1,S5,0.000,0.355
7,S8,S12,0.000,0.450


In [26]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(T4.toarray())
res = pd.DataFrame({"Sentence": sent4, "True topic": topic4, "Cluster": km.labels_}, index=ids4)
res.sort_values("Cluster")

,Sentence,True topic,Cluster
S8,Programmers write code for learning models,ai_tech,0
S7,Deep learning models need large data,ai_tech,0
S6,Machine learning models learn from training data,ai_tech,0
S5,Neural networks learn patterns from large data,ai_tech,0
S12,Fresh vegetables and fruits make healthy food,food,1
S11,The restaurant serves fresh spicy food,food,1
S10,The chef baked a sweet chocolate cake,food,1
S9,The chef cooked fresh pasta with tomato sauce,food,1
S1,The football team won the match yesterday,sports,2
S2,The cricket team won the final match,sports,2


In [27]:
doc_vecs = {}
for topic in ["sports", "ai_tech", "food"]:
    ix = [i for i, t in enumerate(topic4) if t == topic]
    doc_vecs[topic] = {"tfidf_mean": T4.toarray()[ix].mean(0), "w2v_mean": A4[ix].mean(0)}
print("Doc vector dims -> TF-IDF:", doc_vecs["sports"]["tfidf_mean"].shape, "| Word2Vec:", doc_vecs["sports"]["w2v_mean"].shape)
D = np.vstack([doc_vecs[t]["tfidf_mean"] for t in doc_vecs])
pd.DataFrame(cosine_similarity(D), index=list(doc_vecs), columns=list(doc_vecs)).round(3)

Doc vector dims -> TF-IDF: (44,) | Word2Vec: (50,)


,sports,ai_tech,food
sports,1.0,0.0,0.0
ai_tech,0.0,1.0,0.0
food,0.0,0.0,1.0


In [28]:
comp = pd.DataFrame({
    "Level": ["Word (co-occurrence, Q3)", "Word (Word2Vec)", "Sentence (TF-IDF)", "Sentence (Avg W2V)", "Document (mean of sentences)"],
    "Dimension": [len(vocab3), w2v.vector_size, T4.shape[1], A4.shape[1], f"{T4.shape[1]} or {A4.shape[1]}"],
    "Type": ["sparse, count based", "dense, learned", "sparse, lexical", "dense, semantic", "aggregated, dense or sparse"],
    "Captures word order": ["no", "no", "no", "no", "no"],
})
comp

,Level,Dimension,Type,Captures word order
0,"Word (co-occurrence, Q3)",41,"sparse, count based",no
1,Word (Word2Vec),30,"dense, learned",no
2,Sentence (TF-IDF),44,"sparse, lexical",no
3,Sentence (Avg W2V),50,"dense, semantic",no
4,Document (mean of sentences),44 or 50,"aggregated, dense or sparse",no


**Challenges of fixed-length vectors for sentences and documents**

- **Word order lost:** averaging or bag-of-words treats "dog bites man" and "man bites dog" alike.
- **Information bottleneck:** long texts with many ideas get compressed into one vector, diluting details.
- **Averaging dilution:** frequent or unimportant words shift the mean; a single key word can be washed out.
- **Polysemy and context:** one vector per word cannot separate meanings (bank: river vs money).
- **Negation and compositionality:** "good" and "not good" get nearly the same vector.
- **Vocabulary limits:** out-of-vocabulary words are ignored; sparse TF-IDF vectors miss synonyms.
- **Variable length:** a document with 10 or 10,000 words maps to the same size, so scale and structure are lost.
- Better options: contextual encoders such as Sentence-BERT, or doc2vec.